In [65]:
from collections import defaultdict
from sklearn import linear_model
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import math
import gzip
import json
import dateutil.parser
import numpy as np
import random

In [66]:
### Question 1

In [67]:
root = "."
f = gzip.open(root + "/fantasy_10000.json.gz")
dataset = []
for l in f:
    dataset.append(json.loads(l))

In [68]:
print(dataset[0])

{'user_id': '8842281e1d1347389f2ab93d60773d4d', 'book_id': '18245960', 'review_id': 'dfdbb7b0eb5a7e4c26d59a937e2e5feb', 'rating': 5, 'review_text': 'This is a special book. It started slow for about the first third, then in the middle third it started to get interesting, then the last third blew my mind. This is what I love about good science fiction - it pushes your thinking about where things can go. \n It is a 2015 Hugo winner, and translated from its original Chinese, which made it interesting in just a different way from most things I\'ve read. For instance the intermixing of Chinese revolutionary history - how they kept accusing people of being "reactionaries", etc. \n It is a book about science, and aliens. The science described in the book is impressive - its a book grounded in physics and pretty accurate as far as I could tell. Though when it got to folding protons into 8 dimensions I think he was just making stuff up - interesting to think about though. \n But what would happ

In [69]:
def getMaxLen(dataset):
    # Find the longest review (number of characters)
    maxLen = 0
    for item in dataset:
        maxLen = max(maxLen, len(item['review_text']))
        
    return maxLen

In [70]:
def featureQ1(datum, maxLen):
    # Feature vector for one data point
    return [1, (len(datum["review_text"]) / maxLen)]

In [71]:
def Q1(dataset):
    # Implement...
    y = [d["rating"] for d in dataset]
    maxLen = getMaxLen(dataset)

    X = [featureQ1(d, maxLen) for d in dataset]
    theta, residual, rank, s = np.linalg.lstsq(X, y)

    predictions = np.array(X) @ theta
    MSE = mean_squared_error(y, predictions)

    return theta, MSE

In [72]:
### Question 2

In [73]:
for d in dataset:
    t = dateutil.parser.parse(d['date_added'])
    d['parsed_date'] = t

dataset[0]

{'user_id': '8842281e1d1347389f2ab93d60773d4d',
 'book_id': '18245960',
 'review_id': 'dfdbb7b0eb5a7e4c26d59a937e2e5feb',
 'rating': 5,
 'review_text': 'This is a special book. It started slow for about the first third, then in the middle third it started to get interesting, then the last third blew my mind. This is what I love about good science fiction - it pushes your thinking about where things can go. \n It is a 2015 Hugo winner, and translated from its original Chinese, which made it interesting in just a different way from most things I\'ve read. For instance the intermixing of Chinese revolutionary history - how they kept accusing people of being "reactionaries", etc. \n It is a book about science, and aliens. The science described in the book is impressive - its a book grounded in physics and pretty accurate as far as I could tell. Though when it got to folding protons into 8 dimensions I think he was just making stuff up - interesting to think about though. \n But what would 

In [74]:
def featureQ2(datum, maxLen):
    # Implement (should be 1, length feature, day feature, month feature)

    timestamp = datum["parsed_date"]
    day = [0] * 6
    if timestamp.weekday() > 0:
        day[timestamp.weekday() - 1] = 1

    month = [0] * 11
    if timestamp.month > 1:
        month[timestamp.month - 2] = 1

    return [1, len(datum['review_text']) / maxLen] + day + month

In [75]:
def Q2(dataset):

    maxLen = getMaxLen(dataset)

    # Implement (note MSE should be a *number*, not e.g. an array of length 1)
    X2 = [featureQ2(d, maxLen) for d in dataset]
    Y2 = [d["rating"] for d in dataset]
    theta, residual, rank, s = np.linalg.lstsq(X2, Y2)

    predictions = np.array(X2) @ theta
    MSE2 = mean_squared_error(Y2, predictions)

    return X2, Y2, MSE2

In [76]:
### Question 3

In [77]:
def featureQ3(datum, maxLen):
    # Implement
    timestamp = datum["parsed_date"]
    return [1, len(datum["review_text"]) / maxLen, timestamp.weekday(), timestamp.month]

In [78]:
def Q3(dataset):

    maxLen = getMaxLen(dataset)

    X3 = [featureQ3(d, maxLen) for d in dataset]
    Y3 = [d["rating"] for d in dataset]
    theta, residual, rank, s = np.linalg.lstsq(X3, Y3)

    predictions = np.array(X3) @ theta
    MSE3 = mean_squared_error(Y3, predictions)

    return X3, Y3, MSE3

In [79]:
### Question 4

In [80]:
def Q4(dataset):

    # Implement
    # The runner shuffles the data before calling Q4, so don't shuffle again here
    maxLen = getMaxLen(dataset)
    train, test = train_test_split(dataset, test_size=0.5, shuffle=False)

    Y_train = [d["rating"] for d in train]
    Y_test = [d["rating"] for d in test]

    # One-hot encoding (Question 2)
    X2_train = [featureQ2(d, maxLen) for d in train]
    X2_test = [featureQ2(d, maxLen) for d in test]
    theta2, residual, rank, s = np.linalg.lstsq(X2_train, Y_train)
    test_mse2 = mean_squared_error(Y_test, np.array(X2_test) @ theta2)

    # Direct encoding (Question 3)
    X3_train = [featureQ3(d, maxLen) for d in train]
    X3_test = [featureQ3(d, maxLen) for d in test]
    theta3, residual, rank, s = np.linalg.lstsq(X3_train, Y_train)
    test_mse3 = mean_squared_error(Y_test, np.array(X3_test) @ theta3)

    return test_mse2, test_mse3

In [81]:
### Question 5

In [82]:
f = open("beer_50000.json")
datasetB = []
for l in f:
    datasetB.append(eval(l))

datasetB[0]

{'review/appearance': 2.5,
 'beer/style': 'Hefeweizen',
 'review/palate': 1.5,
 'review/taste': 1.5,
 'beer/name': 'Sausa Weizen',
 'review/timeUnix': 1234817823,
 'beer/ABV': 5.0,
 'beer/beerId': '47986',
 'beer/brewerId': '10325',
 'review/timeStruct': {'isdst': 0,
  'mday': 16,
  'hour': 20,
  'min': 57,
  'sec': 3,
  'mon': 2,
  'year': 2009,
  'yday': 47,
  'wday': 0},
 'review/overall': 1.5,
 'review/text': 'A lot of foam. But a lot.\tIn the smell some banana, and then lactic and tart. Not a good start.\tQuite dark orange in color, with a lively carbonation (now visible, under the foam).\tAgain tending to lactic sourness.\tSame for the taste. With some yeast and banana.',
 'user/profileName': 'stcules',
 'review/aroma': 2.0}

In [83]:
y = [d["review/overall"] >= 4 for d in datasetB]

In [84]:
def featureQ5(datum):
    # Implement
    return [1, len(datum["review/text"])]

X = [featureQ5(d) for d in datasetB]

In [85]:
def Q5(dataset, feat_func):

    # Implement
    X = [feat_func(d) for d in dataset]
    y = [d["review/overall"] >= 4 for d in dataset]

    mod2 = linear_model.LogisticRegression(class_weight = 'balanced')
    mod2.fit(X, y)
    pred = mod2.predict(X)

    TP = sum(np.logical_and(pred, y))
    FP = sum(np.logical_and(pred, np.logical_not(y)))
    TN = sum(np.logical_and(np.logical_not(pred), np.logical_not(y)))
    FN = sum(np.logical_and(np.logical_not(pred), y))

    BER = 1 - 0.5*(TP / (TP + FN) + TN / (TN + FP))

    return TP, TN, FP, FN, BER

In [86]:
### Question 6

In [87]:
def Q6(dataset):
    # Implement
    X = [featureQ5(d) for d in dataset]
    y = [d["review/overall"] >= 4 for d in dataset]

    mod2 = linear_model.LogisticRegression(class_weight = 'balanced')
    mod2.fit(X, y)
    pred = mod2.predict(X)
    scores = mod2.decision_function(X)

    scoreslabels = list(zip(scores, y))
    scoreslabels.sort(reverse=True)
    sortedlabels = [x[1] for x in scoreslabels]

    precs = [sum(sortedlabels[:K]) / K for K in [1, 100, 1000, 10000]]

    return precs

In [ ]:
### Question 7

In [ ]:
# Implement (any feature vector which improves performance over Q5)
styles = set(d["beer/style"] for d in datasetB)
styleIndex = {s : i for i, s in enumerate(styles)}

def featureQ7(datum):

    oneHot = [0] * len(styles)
    if datum["beer/style"] in styleIndex:
        oneHot[styleIndex[datum["beer/style"]]] = 1

    return [1, len(datum["review/text"]) / 1000] + oneHot

In [ ]:
y = [d["review/overall"] >= 4 for d in datasetB]
X = [featureQ7(d) for d in datasetB]

mod2 = linear_model.LogisticRegression(class_weight = 'balanced')
mod2.fit(X, y)
pred = mod2.predict(X)

TP = sum(np.logical_and(pred, y))
FP = sum(np.logical_and(pred, np.logical_not(y)))
TN = sum(np.logical_and(np.logical_not(pred), np.logical_not(y)))
FN = sum(np.logical_and(np.logical_not(pred), y))

BER = 1 - 0.5*(TP / (TP + FN) + TN / (TN + FP))
print(BER)